# 3.1c Scaled Auto-Generated Ontology for Text-to-SQL on Spider4SSC: Is the Gain the Ontology or the Curation?

*Formerly notebook 30.*

**Presentation map:** Slide 27 (an ontology defines classes, properties and links), Slides 45–50 (LLMs build the
graph; validation), Slides 59–61 (deterministic checks, LLM repair, "unknown" after three failures).

Notebook 3.1b found a large gain from checking queries against an ontology (+61.5 points over a retry baseline), but that
ontology was **curated by hand**: the RDF schema that ships with Spider4SSC. Building such an ontology is the expensive
part. [NVIDIA Auto Ontology](https://github.com/NVIDIA/auto-ontology) (Apache 2.0) generates one **automatically from
database metadata**: LLM calls describe tables and columns, propose business terms with synonyms, and propose join
links (`SEMANTIC_FK`). Its text-to-SQL agent then checks every `JOIN … ON` against those links before it runs a query.
This notebook asks whether that kind of generated ontology still helps, on the same questions as notebook 3.1b, with
SQL over the original SQLite databases.

See `../research/nvidia-auto-ontology-integration.md` for the assessment behind this design.

## Protocol (frozen before the confidence run; changes are logged at the end)

- **Research question:** does an ontology generated from metadata alone improve execution accuracy over a strong
  schema-only baseline, and does a deterministic join-path check built on it reduce confidently wrong answers?
- **Hypothesis:** with declared foreign keys the gain is small, because Spider's schemas are small and clean.
  Without declared foreign keys, the generated join links matter, and the ontology systems beat the baseline.
- **Data:** Spider4SSC dev, the **563 questions over 19 SQLite databases** used by notebook 3.1b (`wta_1` excluded), with
  the gold **SQL** of each question. Test mode uses 4 databases × 4 questions.
- **Two schema variants:**
  - `fk`: the schema as shipped, with its declared foreign keys;
  - `nofk`: the same schema with every declared foreign key removed, from the prompts and from the ontology builder's
    input. Join links then have to be inferred.
- **Four systems per variant**, same model (`gpt-6-luna`, temperature 0, seed 7) through `smash_batch.py`:

| System | Prompt | After the first query |
| :--- | :--- | :--- |
| (a) **One-shot** | schema DDL with 3 sample values per column | executed as is |
| (b) **Retry** (strong baseline) | same as (a) | an execution error or an empty result is sent back; ≤ 3 retries |
| (c) **Ontology context** | (a) plus the generated ontology: table and column descriptions, business terms with synonyms, join paths | same retry rule as (b) |
| (d) **Ontology + join check** | same as (c) | (b)'s rule plus a deterministic **join-path check** (no LLM): every column-to-column equality between two tables must be a link of the ontology; violations are explained back with the valid paths; ≤ 3 repairs, then **"unknown"** |

  The baseline gets the same raw metadata the ontology builder sees (schema and sample values), as rule 4 of
  `../paper/v2/PLAN.md` §5b requires. The ontology adds only what the builder generated from it.
- **Primary metric:** execution accuracy: the multiset of result rows equals the gold result (values normalised, row
  and column order ignored, as in notebook 3.1b). The verdict is on **(d) − (b)** in each variant, with a paired 95%
  bootstrap interval.
- **Secondary metrics:** confidently wrong rate, "unknown" rate, (c) − (b), accuracy on the questions whose gold query
  joins tables, join violations found, how often the checker would reject a **gold** query (false rejections), and how
  many of the declared foreign keys the builder recovers in the `nofk` variant. The cost of the ontology build is
  reported apart from query time.
- **Threats:** Spider is easy for current models, so the headroom is small; the share of join questions is reported.
  The ontology builder is itself an LLM stage, so its model and cost are reported. The join check is a token-level
  parser, not a full SQL parser (see its cell). Numbers are not comparable with notebook 3.1b's, which used SPARQL.

## Where the ontology comes from

The notebook looks for **frozen Auto Ontology exports** in `data/benchmarks/spider4ssc/auto_ontology/<variant>/<db>.yaml`
(the native Auto Ontology YAML, not Ossie). If every database has one, those are used, and the build needs no API.

If they are missing, the notebook builds a **stand-in**: one structured LLM call per database that does what Auto
Ontology's semantic compilation does (describe tables and columns, propose terms with synonyms, propose join links
from names, types and sample values), and adds the declared foreign keys directly in the `fk` variant. The stand-in is
cached and replayable like every other stage. Its results describe *an LLM-generated ontology*, not Auto Ontology
itself; the summary states which source was used.

To use the real tool:

1. Set `PREPARE_AUTO_ONTOLOGY_INPUTS = True` in the ontology cell and run it once. It writes copies of the 19
   databases **without foreign keys** to `data/benchmarks/spider4ssc/auto_ontology/nofk_sqlite/`.
2. Run Auto Ontology (pinned to commit `cdaf61e`) outside this project, ingest the original databases
   (`database/<db>/<db>.sqlite`) and the stripped copies through its SQLite connector, and compile each one.
3. Export each database in the native format and save it as `auto_ontology/fk/<db>.yaml` or `auto_ontology/nofk/<db>.yaml`.
   Record the commit, the builder model and the build cost in the change log at the end of this notebook.
4. Re-run this notebook. The query-time stages are cached per prompt, so only the systems whose prompt changed are re-run.
   Reading YAML needs `pyyaml`.

The generated ontologies derive from Spider's schemas, so publish them under CC BY-SA 4.0 with Spider attribution.

In [ ]:
# Notebooks live in examples/<task group>/; data/, smash_batch.py and the .venv-* environments stay in examples/.
# Every path below is relative to examples/, so run from there whether the notebook is opened in its own folder or not.
import os, sys
if not os.path.exists("smash_batch.py") and os.path.exists(os.path.join("..", "smash_batch.py")):
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [1]:
import json, os, re, sqlite3, time
from collections import Counter, defaultdict, deque
import numpy as np
from pydantic import BaseModel
from smash_batch import LLM, mode_from_env, wilson, paired_boot

MODE = mode_from_env()
TEST = MODE == "test"
BASE = "data/benchmarks/spider4ssc"
AO_DIR = f"{BASE}/auto_ontology"            # frozen Auto Ontology exports: <variant>/<db>.yaml
llm = LLM(f"{BASE}/cache_30" + ("_test" if TEST else ""))
VARIANTS = ["fk", "nofk"]

dev = [dict(q, qid=f"q{i:03d}") for i, q in enumerate(json.load(open(f"{BASE}/dev.json"))) if q["db_id"] != "wta_1"]
NB17 = f"{BASE}/cache/round0.jsonl"         # notebook 3.1b's question set (563 of the 572), if its cache is present
if os.path.exists(NB17):
    keep = {json.loads(line)["key"] for line in open(NB17)}
    dev = [q for q in dev if q["qid"] in keep]
if TEST:
    dev = [q for db in sorted({q["db_id"] for q in dev})[:4] for q in [x for x in dev if x["db_id"] == db][:4]]
DBS = sorted({q["db_id"] for q in dev})
print(f"{len(dev)} questions over {len(DBS)} databases (mode: {MODE})")

563 questions over 19 databases (mode: batch)


## The schema of each database, read from SQLite

Tables, columns, types, primary keys and declared foreign keys come from SQLite itself (`PRAGMA table_info`,
`PRAGMA foreign_key_list`). A foreign key that names a missing table or column is dropped. Every column gets up to
three distinct sample values: Auto Ontology reads sample values too, and the baseline sees the same ones.

In [2]:
def connect(db, path=None):
    con = sqlite3.connect(f"file:{path or f'{BASE}/database/{db}/{db}.sqlite'}?mode=ro", uri=True)
    con.text_factory = lambda b: b.decode("utf-8", "replace")
    return con

def qi(name):
    return '"' + name.replace('"', '""') + '"'

SCHEMA, DROPPED_FK = {}, 0
for db in DBS:
    con = connect(db)
    names = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")]
    tables = {}
    for t in names:
        cols = con.execute(f"PRAGMA table_info({qi(t)})").fetchall()      # cid, name, type, notnull, default, pk
        tables[t] = [{"name": c[1], "type": c[2] or "", "notnull": bool(c[3]), "pk": c[5],
                      "samples": [r[0] for r in con.execute(
                          f"SELECT DISTINCT {qi(c[1])} FROM {qi(t)} WHERE {qi(c[1])} IS NOT NULL LIMIT 3")]}
                     for c in cols]
    lower = {t.lower(): t for t in tables}
    def col_name(t, c):
        return next((x["name"] for x in tables[t] if x["name"].lower() == (c or "").lower()), None)
    fks = []
    for t in tables:
        for r in con.execute(f"PRAGMA foreign_key_list({qi(t)})"):           # id, seq, table, from, to, ...
            rt = lower.get(r[2].lower())
            if rt is None:
                DROPPED_FK += 1; continue
            to = r[4] or next((x["name"] for x in tables[rt] if x["pk"]), None)
            a, b = col_name(t, r[3]), col_name(rt, to)
            if a is None or b is None:
                DROPPED_FK += 1; continue
            fks.append(((t, a), (rt, b)))
    con.close()
    SCHEMA[db] = {"tables": tables, "fks": sorted(set(fks))}

def show(v):
    if isinstance(v, str):
        v = " ".join(v.split())               # one line: newlines would break the SQL comment
        return "'" + (v[:30] + "…" if len(v) > 30 else v) + "'"
    return repr(v)

def ddl(db, variant):
    out = []
    for t, cols in SCHEMA[db]["tables"].items():
        items = [(f"{c['name']} {c['type']}".rstrip(), "e.g. " + ", ".join(show(v) for v in c["samples"]) if c["samples"] else "")
                 for c in cols]
        pk = [c["name"] for c in sorted(cols, key=lambda c: c["pk"]) if c["pk"]]
        if pk:
            items.append((f"PRIMARY KEY ({', '.join(pk)})", ""))
        if variant == "fk":
            items += [(f"FOREIGN KEY ({a[1]}) REFERENCES {b[0]}({b[1]})", "") for a, b in SCHEMA[db]["fks"] if a[0] == t]
        lines = [f"  {d}{',' if i < len(items) - 1 else ''}" + (f"  -- {note}" if note else "") for i, (d, note) in enumerate(items)]
        out.append(f"CREATE TABLE {t} (\n" + "\n".join(lines) + "\n);")
    return "\n".join(out)

n_t = sum(len(SCHEMA[db]["tables"]) for db in DBS)
n_c = sum(len(c) for db in DBS for c in SCHEMA[db]["tables"].values())
print(f"{n_t} tables, {n_c} columns, {sum(len(SCHEMA[db]['fks']) for db in DBS)} declared foreign keys "
      f"({DROPPED_FK} invalid ones dropped)\n")
print(ddl(DBS[0], "fk"))

77 tables, 396 columns, 60 declared foreign keys (0 invalid ones dropped)

CREATE TABLE battle (
  id INT,  -- e.g. 1, 2, 3
  name TEXT,  -- e.g. 'Battle of Adrianople', 'Battle of Serres', 'Battle of Rusion'
  date TEXT,  -- e.g. '14 April 1205', 'June 1205', '31 January 1206'
  bulgarian_commander TEXT,  -- e.g. 'Kaloyan', 'Unknown', 'Boril'
  latin_commander TEXT,  -- e.g. 'Baldwin I', 'Unknown', 'Thierry de Termond'
  result TEXT,  -- e.g. 'Bulgarian victory', 'Latin victory', 'Two Year Truce'
  PRIMARY KEY (id)
);
CREATE TABLE ship (
  lost_in_battle INT,  -- e.g. 8, 7, 6
  id INT,  -- e.g. 1, 2, 3
  name TEXT,  -- e.g. 'Lettice', 'Bon Accord', 'Mary'
  tonnage TEXT,  -- e.g. 't', '391', '225'
  ship_type TEXT,  -- e.g. 'Brig', '18-gun Brig', '8 gun Brig'
  location TEXT,  -- e.g. 'English Channel', 'SW Approaches', 'Mid-Atlantic'
  disposition_of_ship TEXT,  -- e.g. 'Captured', 'Wrecked', 'Scuttled'
  PRIMARY KEY (id),
  FOREIGN KEY (lost_in_battle) REFERENCES battle(id)
);
CREAT

## Executing queries safely

Queries run on a **read-only** connection with a 20-second limit. Only a single `SELECT` (or `WITH … SELECT`) is
accepted. Results are normalised as in notebook 3.1b: numbers rounded to 4 decimals, each row turned into a sorted
tuple, rows compared as a multiset.

In [3]:
def norm(v):
    if v is None:
        return ""
    if isinstance(v, bytes):
        v = v.decode("utf-8", "replace")
    if isinstance(v, (int, float)) or (isinstance(v, str) and re.fullmatch(r"-?\d+(\.\d+)?", v.strip() or "x")):
        try:
            return f"{round(float(v), 4):g}"
        except Exception:
            pass
    return str(v).strip()

def execute(db, sql, timeout=20, max_rows=100_000):
    if not re.match(r"\s*(SELECT|WITH)\b", sql or "", re.I):
        return "error", "READ-ONLY GATE: only a single SELECT query is allowed"
    con, t0 = connect(db), time.time()
    con.set_progress_handler(lambda: int(time.time() - t0 > timeout), 10_000)
    try:
        rows = con.execute(sql).fetchmany(max_rows)
        return "ok", [tuple(sorted(norm(v) for v in r)) for r in rows]
    except Exception as e:
        msg = f"Timeout: the query ran longer than {timeout} s" if time.time() - t0 > timeout else f"{type(e).__name__}: {str(e)[:200]}"
        return "error", msg
    finally:
        con.close()

same = lambda a, b: Counter(a) == Counter(b)
GOLD = {q["qid"]: execute(q["db_id"], q["sql"]) for q in dev}
bad = [qid for qid, (s, _) in GOLD.items() if s != "ok"]
dev = [q for q in dev if q["qid"] not in bad]
print(f"Gold SQL executed: {len(GOLD) - len(bad)} ok, {len(bad)} failed (dropped) -> {len(dev)} questions; "
      f"{sum(1 for q in dev if not GOLD[q['qid']][1])} gold results are empty")

Gold SQL executed: 563 ok, 0 failed (dropped) -> 563 questions; 0 gold results are empty


## The deterministic join-path check (no LLM)

The check follows Auto Ontology's `join_path_check`: a join must follow a link of the ontology. It works on SQL
tokens:

1. Table names and aliases are collected after every `FROM` and `JOIN` (including comma lists and subqueries). An alias
   reused in several subqueries keeps all its tables, and the check passes if any of them fits.
2. Every equality `x.col = y.col2` between columns of two tables (in `ON` or `WHERE`) is a join condition. Unqualified
   columns are resolved when exactly one table of the query has them.
3. A join condition passes if both columns hold **the same identifier** under the ontology: they are linked directly, or
   through a chain of links (two child tables that both reference the same key may be joined on it).
4. A `JOIN` without `ON` or `USING` is reported.

On a violation the message names the columns and lists the shortest valid join paths between the two tables.
Limitations: `USING`, joins through derived tables and non-equality join conditions are not checked.

In [4]:
KW = {"SELECT", "FROM", "WHERE", "JOIN", "INNER", "LEFT", "RIGHT", "FULL", "OUTER", "CROSS", "NATURAL", "ON", "USING",
      "GROUP", "ORDER", "BY", "LIMIT", "OFFSET", "UNION", "INTERSECT", "EXCEPT", "HAVING", "AS", "AND", "OR", "NOT",
      "IN", "IS", "NULL", "LIKE", "BETWEEN", "EXISTS", "CASE", "WHEN", "THEN", "ELSE", "END", "DISTINCT", "ALL", "ASC",
      "DESC", "WITH"}
TOKEN = re.compile(r"'(?:[^']|'')*'|\"(?:[^\"]|\"\")*\"|`[^`]*`|\[[^\]]*\]|\d+(?:\.\d+)?|[A-Za-z_][\w$]*|<=|>=|<>|!=|==|\S")

def tokens(sql):
    sql = re.sub(r"--[^\n]*|/\*.*?\*/", " ", sql or "", flags=re.S)
    out = []
    for m in TOKEN.finditer(sql):
        t = m.group(0)
        if t[0] in "'\"":
            out.append(("str", t))
        elif t[0] in "`[":
            out.append(("id", t[1:-1]))
        elif t[0].isdigit():
            out.append(("num", t))
        elif t[0].isalpha() or t[0] == "_":
            out.append(("id", t))
        else:
            out.append(("op", t))
    return out

class Links:
    """Join links of one ontology: union-find over (table, column), plus a table graph for suggestions."""
    def __init__(self, db, pairs):
        self.db, self.pairs = db, sorted(set(pairs))
        self.parent = {}
        self.adj = defaultdict(list)
        for a, b in self.pairs:
            self.union(self.key(a), self.key(b))
            self.adj[a[0]].append((b[0], a, b)); self.adj[b[0]].append((a[0], b, a))
    @staticmethod
    def key(tc):
        return (tc[0].lower(), tc[1].lower())
    def find(self, x):
        self.parent.setdefault(x, x)
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]; x = self.parent[x]
        return x
    def union(self, a, b):
        self.parent[self.find(a)] = self.find(b)
    def linked(self, a, b):
        a, b = self.key(a), self.key(b)
        return a in self.parent and b in self.parent and self.find(a) == self.find(b)
    def paths(self, t1, t2, max_len=3):
        """Shortest chains of links between two tables, as text."""
        best, queue = None, deque([(t1, [])])
        seen = {t1: 0}
        found = []
        while queue:
            t, path = queue.popleft()
            if best is not None and len(path) > best:
                break
            if t == t2 and path:
                best = len(path); found.append(path); continue
            if len(path) == max_len:
                continue
            for nt, a, b in self.adj[t]:
                if seen.get(nt, 99) >= len(path) + 1 or nt == t2:
                    seen[nt] = len(path) + 1
                    queue.append((nt, path + [(a, b)]))
        return [", ".join(f"{a[0]}.{a[1]} = {b[0]}.{b[1]}" for a, b in p) for p in found[:3]]

def join_check(db, sql, links):
    """List of violations (empty = the query passes) and the number of join conditions checked."""
    tables = SCHEMA[db]["tables"]
    tabs = {t.lower(): t for t in tables}
    cols = {t: {c["name"].lower(): c["name"] for c in cs} for t, cs in tables.items()}
    toks = tokens(sql)
    up = [t.upper() if k == "id" else t for k, t in toks]
    alias, used, viol = defaultdict(set), [], []
    for i in range(len(toks)):
        if up[i] not in ("FROM", "JOIN"):
            continue
        j = i + 1
        while j < len(toks) and toks[j][0] == "id" and toks[j][1].lower() in tabs and up[j] not in KW:
            t = tabs[toks[j][1].lower()]
            used.append(t); alias[t.lower()].add(t)
            k = j + 1
            if k < len(toks) and up[k] == "AS":
                k += 1
            if k < len(toks) and toks[k][0] == "id" and up[k] not in KW:
                alias[toks[k][1].lower()].add(t); k += 1
            if up[i] == "JOIN" and (k >= len(toks) or up[k] not in ("ON", "USING")):
                viol.append(f"JOIN WITHOUT CONDITION: `JOIN {t}` has no ON clause; join it on one of its links.")
            if up[i] == "FROM" and k < len(toks) and toks[k][1] == ",":
                j = k + 1; continue
            break

    def column(ts, c):
        return {(t, cols[t][c.lower()]) for t in ts if c.lower() in cols[t]}
    def left_ref(e):                       # column reference that ends just before index e
        if e >= 3 and toks[e - 1][0] == "id" and toks[e - 2][1] == "." and toks[e - 3][0] == "id":
            q = toks[e - 3][1].lower()
            return column(alias.get(q) or ({tabs[q]} if q in tabs else set()), toks[e - 1][1]), q
        if e >= 1 and toks[e - 1][0] == "id" and up[e - 1] not in KW and (e < 2 or toks[e - 2][1] != "."):
            hit = column(set(used), toks[e - 1][1])
            return (hit if len(hit) == 1 else set()), None
        return set(), None
    def right_ref(s):                      # column reference that starts at index s
        if s + 2 < len(toks) and toks[s][0] == "id" and toks[s + 1][1] == "." and toks[s + 2][0] == "id":
            q = toks[s][1].lower()
            return column(alias.get(q) or ({tabs[q]} if q in tabs else set()), toks[s + 2][1]), q
        if s < len(toks) and toks[s][0] == "id" and up[s] not in KW and (s + 1 >= len(toks) or toks[s + 1][1] not in (".", "(")):
            hit = column(set(used), toks[s][1])
            return (hit if len(hit) == 1 else set()), None
        return set(), None

    checked = 0
    for e in range(1, len(toks) - 1):
        if toks[e][1] not in ("=", "=="):
            continue
        L, ql = left_ref(e)
        R, qr = right_ref(e + 1)
        if not L or not R or (ql is not None and ql == qr):
            continue                       # not a column = column condition between two table references
        pairs = [(a, b) for a in L for b in R if a != b]
        if not pairs:
            continue
        checked += 1
        if any(links.linked(a, b) for a, b in pairs):
            continue
        a, b = pairs[0]
        sugg = links.paths(a[0], b[0])
        viol.append(f"JOIN PATH: `{a[0]}.{a[1]} = {b[0]}.{b[1]}` joins two columns that the ontology does not link. "
                    + (f"Valid join paths between {a[0]} and {b[0]}: " + " | ".join(sugg) if sugg
                       else f"The ontology has no join path between {a[0]} and {b[0]}."))
    return viol, checked

DECLARED = {db: Links(db, SCHEMA[db]["fks"]) for db in DBS}
joins_gold = {q["qid"]: join_check(q["db_id"], q["sql"], DECLARED[q["db_id"]]) for q in dev}
JOIN_Q = {qid for qid, (_, n) in joins_gold.items() if n > 0}
rej = [qid for qid, (v, _) in joins_gold.items() if v]
print(f"Gold queries with a join condition: {len(JOIN_Q)} of {len(dev)} ({len(JOIN_Q) / len(dev):.0%})")
print(f"Gold queries the check would reject with the declared foreign keys as links: {len(rej)} ({len(rej) / len(dev):.1%})")
for qid in rej[:3]:
    q = next(x for x in dev if x["qid"] == qid)
    print(f"  [{q['db_id']}] {q['sql']}\n    -> {joins_gold[qid][0][0][:200]}")

Gold queries with a join condition: 180 of 563 (32%)
Gold queries the check would reject with the declared foreign keys as links: 6 (1.1%)
  [flight_2] SELECT count(*) FROM AIRLINES AS T1 JOIN FLIGHTS AS T2 ON T2.Airline  =  T1.uid WHERE T1.Airline  =  "United Airlines" AND T2.DestAirport  =  "ASY"
    -> JOIN PATH: `flights.Airline = airlines.uid` joins two columns that the ontology does not link. The ontology has no join path between flights and airlines.
  [flight_2] SELECT count(*) FROM AIRLINES AS T1 JOIN FLIGHTS AS T2 ON T2.Airline  =  T1.uid WHERE T1.Airline  =  "United Airlines" AND T2.DestAirport  =  "ASY"
    -> JOIN PATH: `flights.Airline = airlines.uid` joins two columns that the ontology does not link. The ontology has no join path between flights and airlines.
  [flight_2] SELECT count(*) FROM AIRLINES AS T1 JOIN FLIGHTS AS T2 ON T2.Airline  =  T1.uid WHERE T1.Airline  =  "United Airlines" AND T2.SourceAirport  =  "AHD"
    -> JOIN PATH: `flights.Airline = airlines.uid` j

## The ontology: Auto Ontology export, or the LLM-built stand-in

`load_auto_ontology` reads the native Auto Ontology YAML (the `AutoOntologyModelDocument` of
`auto_ontology/server/model_interchange/schemas.py` at commit `cdaf61e`): table and column descriptions from
`data_layer`, declared foreign keys and joins, terms with the tables they represent, and the `semantic_fks` resolved
through the terms' column attributes. This reader has not yet been tested on a real export.

The stand-in (`build_onto` stage) sends each database's schema and sample values, in the variant's form, to one
structured call. Proposed links whose tables or columns do not exist are dropped. In the `fk` variant the declared
foreign keys are added, as Auto Ontology uses a declared foreign key directly.

In [5]:
PREPARE_AUTO_ONTOLOGY_INPUTS = False      # True: write FK-free copies of the databases for an Auto Ontology build

def write_fk_free_copy(db, out):
    """Same tables, columns, primary keys and rows; no foreign keys. Rows are copied by SQLite, byte for byte."""
    if os.path.exists(out):
        os.remove(out)
    dst = sqlite3.connect(out)
    for t, cs in SCHEMA[db]["tables"].items():
        defs = [f"{qi(c['name'])} {c['type']}".rstrip() + (" NOT NULL" if c["notnull"] else "") for c in cs]
        pk = [c["name"] for c in sorted(cs, key=lambda c: c["pk"]) if c["pk"]]
        if pk:
            defs.append("PRIMARY KEY (" + ", ".join(qi(c) for c in pk) + ")")
        dst.execute(f"CREATE TABLE {qi(t)} ({', '.join(defs)})")
    dst.execute("ATTACH DATABASE ? AS src", (f"{BASE}/database/{db}/{db}.sqlite",))
    for t in SCHEMA[db]["tables"]:
        dst.execute(f"INSERT INTO main.{qi(t)} SELECT * FROM src.{qi(t)}")
    dst.commit(); dst.execute("DETACH DATABASE src"); dst.close()

if PREPARE_AUTO_ONTOLOGY_INPUTS:
    os.makedirs(f"{AO_DIR}/nofk_sqlite", exist_ok=True)
    for db in DBS:
        write_fk_free_copy(db, f"{AO_DIR}/nofk_sqlite/{db}.sqlite")
    print(f"Wrote {len(DBS)} FK-free copies to {AO_DIR}/nofk_sqlite/")

def match_table(db, t):
    """Table of the SQLite schema with this name (case-insensitive; a schema prefix such as `main.` is ignored)."""
    tabs = {x.lower(): x for x in SCHEMA[db]["tables"]}
    return tabs.get((t or "").split(".")[-1].strip('"`').lower())

def match_tc(db, t, c):
    t = match_table(db, t)
    if t is None:
        return None
    c = next((x["name"] for x in SCHEMA[db]["tables"][t] if x["name"].lower() == (c or "").strip('"`').lower()), None)
    return (t, c) if c else None

def load_auto_ontology(db, variant, path):
    if path.endswith((".yaml", ".yml")):
        import yaml
        doc = yaml.safe_load(open(path))
    else:
        doc = json.load(open(path))
    assert "data_layer" in doc, f"{path}: expected the native Auto Ontology export, not Ossie"
    tab_by_id, col_by_id, onto = {}, {}, {"source": "Auto Ontology export", "tables": {}, "columns": {}, "terms": [], "links": []}
    for d in doc["data_layer"].get("databases", []):
        for s in d.get("schemas", []):
            for t in s.get("tables", []):
                tname = match_table(db, t.get("name"))
                if tname is None:
                    continue
                tab_by_id[t["id"]] = tname; onto["tables"][tname] = t.get("description", "")
                for c in t.get("columns", []):
                    tc = match_tc(db, tname, c.get("name"))
                    if tc:
                        col_by_id[c["id"]] = tc; onto["columns"][tc] = c.get("description", "")
    links = []
    for fk in doc["data_layer"].get("foreign_keys", []):
        if fk["source_column_id"] in col_by_id and fk["target_column_id"] in col_by_id:
            links.append((col_by_id[fk["source_column_id"]], col_by_id[fk["target_column_id"]]))
    for j in doc["data_layer"].get("joins", []):
        ts, tt = tab_by_id.get(j["source_table_id"]), tab_by_id.get(j["target_table_id"])
        for jc in j.get("join_columns", []):
            a, b = (match_tc(db, ts, jc.get("source")) if ts else None), (match_tc(db, tt, jc.get("target")) if tt else None)
            if a and b:
                links.append((a, b))
    sem = doc.get("semantic_layer", {})
    attr_col = {}
    for term in sem.get("terms", []):
        attrs = term.get("columns_attributes", [])
        attr_col.update({a["id"]: a.get("column_id") for a in attrs})
        onto["terms"].append({"name": term.get("name", ""), "synonyms": [], "description": term.get("description", ""),
                              "tables": [tab_by_id[r] for r in term.get("represents", []) if r in tab_by_id]})
    for sf in sem.get("semantic_fks", []):
        a, b = col_by_id.get(sf.get("column_id")), col_by_id.get(attr_col.get(sf.get("column_attribute_id")))
        if sf.get("use_in_join_paths", True) and a and b:
            links.append((a, b))
    onto["links"] = sorted({tuple(sorted(p)) for p in links if p[0] != p[1]})
    return onto

class TableDesc(BaseModel):
    table: str
    description: str
class ColumnDesc(BaseModel):
    column: str                 # "table.column"
    description: str
class Term(BaseModel):
    name: str
    synonyms: list[str]
    description: str
    tables: list[str]
class Link(BaseModel):
    source: str                 # "table.column" (the referencing column)
    target: str                 # "table.column" (the key it references)
class Ontology(BaseModel):
    tables: list[TableDesc]
    columns: list[ColumnDesc]
    terms: list[Term]
    links: list[Link]

BUILD_SYS = ("You build a semantic layer (an ontology) over a relational database from its metadata only: the schema "
             "and a few sample values per column. Produce:\n"
             "1. a one-sentence business description of every table, and of every column (`table.column`);\n"
             "2. the business terms (concepts a user would ask about), each with synonyms and the tables that represent it;\n"
             "3. the join links: every pair of columns that hold the same identifier, so that rows of the two tables can "
             "be joined (`source` = the referencing column, `target` = the key it references). Include declared foreign "
             "keys and any undeclared ones you can infer from names, types and sample values. Do not link columns that "
             "merely share a generic name such as `name` or `id` unless they really hold the same identifier.\n"
             "Use the exact table and column names of the schema.")

def split_tc(s):
    s = (s or "").strip().strip("`")
    return s.rsplit(".", 1) if "." in s else (None, s)

def from_standin(db, variant, parsed):
    onto = {"source": "LLM-built stand-in", "tables": {}, "columns": {}, "terms": [], "links": [], "dropped": 0}
    if parsed:
        for t in parsed["tables"]:
            name = match_table(db, t["table"])
            if name:
                onto["tables"][name] = t["description"]
        for c in parsed["columns"]:
            tc = match_tc(db, *split_tc(c["column"]))
            if tc:
                onto["columns"][tc] = c["description"]
        onto["terms"] = [{**t, "tables": [x for x in (match_table(db, n) for n in t["tables"]) if x]} for t in parsed["terms"]]
        for l in parsed["links"]:
            a, b = match_tc(db, *split_tc(l["source"])), match_tc(db, *split_tc(l["target"]))
            if a and b and a != b:
                onto["links"].append((a, b))
            else:
                onto["dropped"] += 1
    if variant == "fk":
        onto["links"] += SCHEMA[db]["fks"]
    onto["links"] = sorted({tuple(sorted(p)) for p in onto["links"]})
    return onto

ONTO, missing = {}, []
for v in VARIANTS:
    for db in DBS:
        path = next((f"{AO_DIR}/{v}/{db}.{ext}" for ext in ("yaml", "yml", "json") if os.path.exists(f"{AO_DIR}/{v}/{db}.{ext}")), None)
        if path:
            ONTO[(v, db)] = load_auto_ontology(db, v, path)
        else:
            missing.append((v, db))
if missing:
    if len(missing) < 2 * len(DBS):
        print(f"Auto Ontology exports found for {2 * len(DBS) - len(missing)} of {2 * len(DBS)}; the rest use the stand-in.")
    reqs = {f"{v}|{db}": LLM.body([{"role": "system", "content": BUILD_SYS},
                                   {"role": "user", "content": ddl(db, v)}], max_tokens=8000, schema=Ontology)
            for v, db in missing}
    built = llm.run("build_onto", reqs, parse=Ontology)
    for v, db in missing:
        ONTO[(v, db)] = from_standin(db, v, built[f"{v}|{db}"]["parsed"])
SOURCE = sorted({o["source"] for o in ONTO.values()})
LINKS = {k: Links(k[1], o["links"]) for k, o in ONTO.items()}

print("Ontology source:", " + ".join(SOURCE))
for v in VARIANTS:
    os_ = [ONTO[(v, db)] for db in DBS]
    print(f"[{v}] {sum(len(o['terms']) for o in os_)} terms, {sum(len(o['links']) for o in os_)} join links, "
          f"{sum(len(o['columns']) for o in os_)} column descriptions, {sum(o.get('dropped', 0) for o in os_)} invalid links dropped")
# Foreign-key recovery in the nofk variant: is every declared FK (as a column pair) linked by the generated ontology?
tp = sum(LINKS[("nofk", db)].linked(a, b) for db in DBS for a, b in SCHEMA[db]["fks"])
n_fk = sum(len(SCHEMA[db]["fks"]) for db in DBS)
gen = [(db, a, b) for db in DBS for a, b in ONTO[("nofk", db)]["links"]]
prec = sum(DECLARED[db].linked(a, b) for db, a, b in gen)
print(f"[nofk] declared foreign keys recovered: {tp} of {n_fk} ({tp / max(n_fk, 1):.0%}); "
      f"generated links that match a declared FK chain: {prec} of {len(gen)} ({prec / max(len(gen), 1):.0%})")
for v in VARIANTS:
    r = [q for q in dev if join_check(q["db_id"], q["sql"], LINKS[(v, q["db_id"])])[0]]
    print(f"[{v}] gold queries the join check would reject with this ontology: {len(r)} ({len(r) / len(dev):.1%})")

Ontology source: LLM-built stand-in
[fk] 111 terms, 62 join links, 396 column descriptions, 0 invalid links dropped
[nofk] 111 terms, 59 join links, 396 column descriptions, 0 invalid links dropped
[nofk] declared foreign keys recovered: 57 of 60 (95%); generated links that match a declared FK chain: 57 of 59 (97%)
[fk] gold queries the join check would reject with this ontology: 0 (0.0%)
[nofk] gold queries the join check would reject with this ontology: 10 (1.8%)


## The ontology as the model sees it

In [6]:
def onto_text(v, db):
    o = ONTO[(v, db)]
    lines = [f"Ontology of the database ({o['source']}, generated from its metadata; it can be incomplete):", "Business terms:"]
    for t in o["terms"]:
        syn = f" (synonyms: {', '.join(t['synonyms'])})" if t.get("synonyms") else ""
        lines.append(f"- {t['name']}{syn} -> {', '.join(t['tables']) or '?'}: {t['description']}")
    lines.append("Tables and columns:")
    for t, cs in SCHEMA[db]["tables"].items():
        lines.append(f"- {t}: {o['tables'].get(t, '')}")
        lines += [f"  - {c['name']}: {o['columns'][(t, c['name'])]}" for c in cs if o["columns"].get((t, c["name"]))]
    lines.append("Join paths (the valid ways to join tables):")
    lines += [f"- {a[0]}.{a[1]} = {b[0]}.{b[1]}" for a, b in o["links"]] or ["- (none)"]
    return "\n".join(lines)

print(onto_text("nofk", DBS[0]))

Ontology of the database (LLM-built stand-in, generated from its metadata; it can be incomplete):
Business terms:
- Battle (synonyms: engagement, military battle) -> battle: A named military engagement with a date, commanders, and an outcome.
- Ship (synonyms: vessel, boat) -> ship: A named vessel with recorded characteristics and disposition.
- Casualty incident (synonyms: casualty event, death incident) -> death: An incident recording the number of people killed or injured and associated context.
- Battle outcome (synonyms: result, victory, settlement) -> battle: The recorded outcome or settlement of a battle.
- Ship loss (synonyms: lost vessel, ship disposition) -> ship, battle: A ship's loss event, including its associated battle and recorded disposition.
- Casualties (synonyms: killed, injured, deaths and injuries) -> death: Counts of people killed and injured in a casualty incident.
Tables and columns:
- battle: Records battles, including their dates, commanders, and outcomes.
  

## Round 0: the first query

(a) and (b) share one first query per variant, and (c) and (d) share another.

In [7]:
SYS = ("You translate questions into a single SQLite SELECT query over the database below. Return only the query in "
       "one ```sql code block.\n\nSchema (with sample values):\n{ddl}")
SYSTEMS = ["one-shot", "retry", "onto", "onto-check"]
PROMPT_OF = {"one-shot": "base", "retry": "base", "onto": "onto", "onto-check": "onto"}

def first_messages(v, kind, q):
    s = SYS.format(ddl=ddl(q["db_id"], v))
    if kind == "onto":
        s += "\n\n" + onto_text(v, q["db_id"])
    return [{"role": "system", "content": s}, {"role": "user", "content": q["question"]}]

def extract_sql(text):
    m = re.findall(r"```(?:sql|sqlite)?\s*(.*?)```", text or "", re.S | re.I)
    return (m[-1] if m else text or "").strip().rstrip(";").strip()

r0 = llm.run("round0", {f"{v}|{kind}|{q['qid']}": LLM.body(first_messages(v, kind, q), max_tokens=800)
                        for v in VARIANTS for kind in ("base", "onto") for q in dev})
STATE = {}
for v in VARIANTS:
    for q in dev:
        for s in SYSTEMS:
            kind = PROMPT_OF[s]; text = r0[f"{v}|{kind}|{q['qid']}"]["text"]
            STATE[(v, s, q["qid"])] = {"query": extract_sql(text), "repairs": 0, "final": None, "log": [],
                                       "msgs": first_messages(v, kind, q) + [{"role": "assistant", "content": text}]}
print(f"Round 0: {len(r0)} queries")

[round0] 2,252 new requests -> batch (0 cached)


[round0] resuming: 1 running, 0 queued


[round0] batch batch_6aeac8242f588190910aec060d7742a5 cancelled: 2,250 ok, 0 failed


[round0] 1 min: 0 running, 0 queued, 2,252 results cached


[round0] re-submitting 2 failed request(s)


[round0] uploaded input file 1/1 (2 requests, 0 MB)


... (56 outputs left out)

[round0] 316 min: 0 running, 0 queued, 2,252 results cached


[round0] done in 316.4 min


[round0] done in 316.9 min


Round 0: 2252 queries


## Rounds 1–3: retry (b, c) and join-checked repair (d), one Batch job per round

In [8]:
MAX_REPAIRS = 3
QS = {q["qid"]: q for q in dev}

def feedback(v, s, q, st):
    """None if the system accepts the current query, else (message, still_failing_check)."""
    status, out = execute(q["db_id"], st["query"])
    st["exec"] = (status, out)
    if s == "one-shot":
        return None
    if s == "onto-check":
        viol, _ = join_check(q["db_id"], st["query"], LINKS[(v, q["db_id"])])
        st["log"].append([x.split(":")[0] for x in viol])
        if viol:
            return ("The query does not follow the ontology's join paths:\n- " + "\n- ".join(viol) +
                    "\nFix these problems and return the corrected query.", True)
    if status != "ok":
        return (f"The query failed: {out}. Fix it and return the corrected query.", s == "onto-check")
    if not out:
        return ("The query returned no rows. Check it and return a corrected query.", False)
    return None

for rnd in range(MAX_REPAIRS + 1):
    reqs = {}
    for (v, s, qid), st in STATE.items():
        if st["final"] is not None:
            continue
        fb = feedback(v, s, QS[qid], st)
        if fb is None:
            st["final"] = "query"
        elif st["repairs"] == MAX_REPAIRS:
            st["final"] = "unknown" if fb[1] else "query"
        else:
            st["msgs"].append({"role": "user", "content": fb[0]})
            reqs[f"{v}|{s}|{qid}"] = LLM.body(st["msgs"], max_tokens=800)
    if not reqs:
        break
    res = llm.run(f"repair{rnd + 1}", reqs)
    for k, r in res.items():
        v, s, qid = k.split("|")
        st = STATE[(v, s, qid)]
        st["msgs"].append({"role": "assistant", "content": r["text"]})
        st["query"] = extract_sql(r["text"]); st["repairs"] += 1
    print(f"Round {rnd + 1}: {len(reqs)} repairs")

[repair1] 36 new requests -> batch (0 cached)


[repair1] uploaded input file 1/1 (36 requests, 0 MB)


[repair1] started batch batch_6aec8158cd708190b0dce61dcd875c20


[repair1] 21 min: 1 running, 0 queued, 0 results cached


[repair1] 27 min: 1 running, 0 queued, 0 results cached


[repair1] 49 min: 1 running, 0 queued, 0 results cached


... (169 outputs left out)

  [repair3] poll of batch_6aed3940358c819096d09b006ca5cb5d failed (APIConnectionError); retrying


[repair3] batch batch_6aed3940358c819096d09b006ca5cb5d completed: 1 ok, 0 failed


[repair3] done in 140.8 min


Round 3: 1 repairs


## Results

In [9]:
def outcome(v, s, qid):
    st = STATE[(v, s, qid)]
    if st["final"] == "unknown":
        return "unknown"
    status, out = st.get("exec") or execute(QS[qid]["db_id"], st["query"])
    return "correct" if status == "ok" and same(out, GOLD[qid][1]) else "wrong"

OUT = {k: outcome(*k) for k in STATE}
acc = lambda v, s, qs=None: [float(OUT[(v, s, q["qid"])] == "correct") for q in (qs or dev)]
RES, DIFF = {}, {}
joinq = [q for q in dev if q["qid"] in JOIN_Q]
for v in VARIANTS:
    print(f"\n[{v}] {'system':<12}{'correct [95% CI]':>22}{'wrong':>9}{'unknown':>10}{'join qs':>10}{'avg repairs':>13}")
    for s in SYSTEMS:
        c = Counter(OUT[(v, s, q["qid"])] for q in dev); n = len(dev)
        lo, hi = wilson(c["correct"], n)
        RES[(v, s)] = (c["correct"], c["wrong"], c["unknown"], n, lo, hi)
        jq = np.mean(acc(v, s, joinq)) if joinq else float("nan")
        print(f"     {s:<12}{f'{c['correct'] / n:.1%} [{lo:.0%}-{hi:.0%}]':>22}{c['wrong'] / n:>9.1%}"
              f"{c['unknown'] / n:>10.1%}{jq:>10.1%}{np.mean([STATE[(v, s, q['qid'])]['repairs'] for q in dev]):>13.2f}")
    for a, b in [("one-shot", "retry"), ("retry", "onto"), ("retry", "onto-check"), ("onto", "onto-check")]:
        DIFF[(v, a, b)] = paired_boot(acc(v, a), acc(v, b))
        dm, lo, hi = DIFF[(v, a, b)]
        print(f"     {b} - {a}: {dm:+.1%} [{lo:+.1%}, {hi:+.1%}]")
    kinds = Counter(k for q in dev for lst in STATE[(v, "onto-check", q["qid"])]["log"] for k in lst)
    print(f"     join-check findings (all rounds): {dict(kinds.most_common())}")
for s in SYSTEMS:
    dm, lo, hi = paired_boot(acc("fk", s), acc("nofk", s))
    print(f"{s}: nofk - fk {dm:+.1%} [{lo:+.1%}, {hi:+.1%}]")


[fk] system            correct [95% CI]    wrong   unknown   join qs  avg repairs
     one-shot           88.6% [86%-91%]    11.4%      0.0%     84.4%         0.00
     retry              89.2% [86%-91%]    10.8%      0.0%     84.4%         0.01
     onto               90.1% [87%-92%]     9.9%      0.0%     84.4%         0.01
     onto-check         90.1% [87%-92%]     9.9%      0.0%     84.4%         0.01
     retry - one-shot: +0.5% [+0.0%, +1.2%]
     onto - retry: +0.9% [-0.2%, +2.0%]
     onto-check - retry: +0.9% [-0.2%, +2.0%]
     onto-check - onto: +0.0% [+0.0%, +0.0%]
     join-check findings (all rounds): {'JOIN WITHOUT CONDITION': 1}

[nofk] system            correct [95% CI]    wrong   unknown   join qs  avg repairs
     one-shot           89.0% [86%-91%]    11.0%      0.0%     84.4%         0.00
     retry              89.5% [87%-92%]    10.5%      0.0%     84.4%         0.01
     onto               89.5% [87%-92%]    10.5%      0.0%     83.9%         0.01
     onto-chec

### A repair the join check made possible

In [10]:
for v in VARIANTS:
    ex = next((q for q in dev if OUT[(v, "onto-check", q["qid"])] == "correct" and OUT[(v, "onto", q["qid"])] != "correct"
               and any(STATE[(v, "onto-check", q["qid"])]["log"])), None)
    if not ex:
        print(f"[{v}] no question where only the join check led to a correct answer"); continue
    st = STATE[(v, "onto-check", ex["qid"])]
    print(f"[{v}] [{ex['db_id']}] {ex['question']}\nFirst query:\n{extract_sql(st['msgs'][2]['content'])}\n")
    for m in st["msgs"][3:]:
        print(("CHECK ->  " if m["role"] == "user" else "REPAIRED -> ") + (m["content"] if m["role"] == "user" else extract_sql(m["content"]))[:500], "\n")

[fk] no question where only the join check led to a correct answer
[nofk] no question where only the join check led to a correct answer


## Cost

`build_onto` is the stand-in ontology build (absent when every Auto Ontology export is present); the other stages
are query time.

In [11]:
LEDGER = llm.print_ledger()

stage                           calls     input tok   output tok      USD  via Batch
build_onto                         38        39,461       42,722     0.01       100%
repair1                            36        52,434        1,209     0.00       100%
repair2                             1         2,927           91     0.00       100%
repair3                             1         3,103           91     0.00       100%
round0                          2,252     2,133,117       76,194     0.13       100%
(embeddings)                        0             0            0     0.00         0%
TOTAL                           2,328     2,231,042      120,307     0.14


In [12]:
print("=" * 100)
print(f"RESULTS SUMMARY — Notebook 3.1c: auto-generated ontology for text-to-SQL on Spider4SSC, {len(dev)} questions / "
      f"{len(DBS)} databases, gpt-6-luna")
print(f"Ontology source: {' + '.join(SOURCE)}; {len(JOIN_Q) / len(dev):.0%} of the gold queries join tables")
print("=" * 100)
for v in VARIANTS:
    print(f"[{'declared foreign keys' if v == 'fk' else 'foreign keys removed'}]")
    for s in SYSTEMS:
        c, w, u, n, lo, hi = RES[(v, s)]
        print(f"  {s:<11} correct {c / n:.1%} [{lo:.0%}-{hi:.0%}] | confidently wrong {w / n:.1%} | unknown {u / n:.1%}")
    for a, b in [("retry", "onto"), ("retry", "onto-check")]:
        dm, lo, hi = DIFF[(v, a, b)]
        print(f"  {b} - {a}: {dm:+.1%} [{lo:+.1%}, {hi:+.1%}]" + ("  resolvable" if lo > 0 or hi < 0 else "  not resolvable")
              + ("   <- primary" if b == "onto-check" else ""))
print(f"[nofk] declared foreign keys recovered by the generated ontology: {tp} of {n_fk}")
print("\nLecture message (Slides 45-50, 59-61): an ontology an LLM generates from metadata is only worth what it adds"
      "\nover a baseline that already sees the schema, sample values and error feedback. Compare the two variants: where"
      "\nthe schema already declares its joins there is little left to add; where it does not, the generated join links"
      "\nand the check built on them are what the ontology contributes.")

RESULTS SUMMARY — Notebook 3.1c: auto-generated ontology for text-to-SQL on Spider4SSC, 563 questions / 19 databases, gpt-6-luna
Ontology source: LLM-built stand-in; 32% of the gold queries join tables
[declared foreign keys]
  one-shot    correct 88.6% [86%-91%] | confidently wrong 11.4% | unknown 0.0%
  retry       correct 89.2% [86%-91%] | confidently wrong 10.8% | unknown 0.0%
  onto        correct 90.1% [87%-92%] | confidently wrong 9.9% | unknown 0.0%
  onto-check  correct 90.1% [87%-92%] | confidently wrong 9.9% | unknown 0.0%
  onto - retry: +0.9% [-0.2%, +2.0%]  not resolvable
  onto-check - retry: +0.9% [-0.2%, +2.0%]  not resolvable   <- primary
[foreign keys removed]
  one-shot    correct 89.0% [86%-91%] | confidently wrong 11.0% | unknown 0.0%
  retry       correct 89.5% [87%-92%] | confidently wrong 10.5% | unknown 0.0%
  onto        correct 89.5% [87%-92%] | confidently wrong 10.5% | unknown 0.0%
  onto-check  correct 88.3% [85%-91%] | confidently wrong 11.5% | unknown 0

## Change log

- 2026-11-04: notebook written, protocol frozen; not yet run. The Auto Ontology export reader is untested; the first
  run uses the stand-in unless exports are added to `data/benchmarks/spider4ssc/auto_ontology/`.